In [ ]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)

%pwd




In [ ]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
import traffic.utils.distribution_utils as du
pd.set_option('display.max_columns', None)

# Process the real count data from Udot


## Day counts

In [ ]:
count_df = pd.read_csv('data/vehicle_counts/210_real_count_data.csv')
count_df.columns = [i.lower() for i in count_df.columns]
count_df['date'] = pd.to_datetime(count_df['date'], format="%m/%d/%y")

hr_cols = list(count_df.columns[5:])
day_counts = count_df.groupby(by='date', as_index=False)[hr_cols].sum()

day_counts['month'] = day_counts['date'].dt.month
day_counts['week'] = day_counts['date'].dt.isocalendar().week
day_counts['weekday'] = day_counts['date'].dt.weekday  # 0=Monday, 6=Sunday
day_counts = day_counts[['date', 'month','week', 'weekday'] + hr_cols]
day_counts['decile'] = np.ceil(day_counts.total.rank(pct=True)*10)
day_counts['percentile'] = np.ceil(day_counts.total.rank(pct=True)*100)
day_counts['pct'] = day_counts.total.rank(pct=True)

day_counts.head(3)



In [ ]:
sns.ecdfplot(day_counts.total)
plt.grid()

In [ ]:
# box plots by day of week
sns.boxplot(data=day_counts, x='weekday', y='total')

# box plot total
sns.boxplot(day_counts.total)

# Weekly average

In [ ]:
agg_dict = {col: 'mean' for col in hr_cols}
agg_dict['weekday'] = 'count'
week_avgs = day_counts.groupby(by='week', as_index=False).agg(agg_dict)

week_avgs['month'] = (week_avgs['week']+3)/4
week_avgs['days_recorded'] = week_avgs.weekday.rename('days_recorded')
week_avgs.drop(columns = 'weekday', inplace=True)
week_avgs = week_avgs[['week', 'month','days_recorded']+hr_cols]

week_avgs['adj_total'] = 7*(week_avgs['total']/week_avgs['days_recorded']) # we got some wack shit going on here
week_avgs.head(3)




In [ ]:
# week_avgs.days_recorded.value_counts()

# week_avgs.sort_values(by='adj_total', ascending =False)

# week_avgs.sort_values(by='days_recorded')

In [ ]:
sns.barplot(data=week_avgs, x='month', y='total')
# Set tick every 4 weeks
ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(4))

# hourly long - The buisness! 

## Takes the day counts and makes it long

In [ ]:
# Select all columns that start with "h"
hour_cols = [col for col in day_counts.columns if col.startswith('h')]

# Melt into long format
df_long = day_counts.melt(
    id_vars=[col for col in day_counts.columns if col not in hour_cols],
    value_vars=hour_cols,
    var_name='hour_str',
    value_name='counts'
)

# Convert 'h0000' → 0, 'h2100' → 21, etc.
df_long['hour'] = df_long['hour_str'].str[1:3].astype(int)
slim_hours = df_long[['hour','counts']]
slim_hours = slim_hours.loc[slim_hours.hour.between(5,21)] # <-- filter out hours before 5 and after 9(21)

slim_hours['hourly_percentile'] = (
    slim_hours.groupby('hour')['counts']
    .transform(lambda x: np.ceil(x.rank(pct=True)*100))
)
print('slim_hours is fully udot data ')
display(slim_hours.head())

# expected_counts is the pivoted version of slim hours
expected_counts = slim_hours.pivot_table(index='hour', columns='hourly_percentile', values='counts')
# expected_counts results in some nuls, i used a rolling average fill
expected_counts_filled = expected_counts.apply(
    lambda row: row.fillna(row.rolling(window=3, center=True, min_periods=1).mean()),
    axis=1
)



## Expected hourly counts --> expected secound counts, using linear interpolation 

In [ ]:
hourly_df = expected_counts_filled.copy()
og_len = len(hourly_df)

# still low low count but the index is spaced out more
hourly_df.index = np.arange(0, og_len*3600, 3600)  # Index in seconds from 0 to 61200

# this produces all the rows 
second_index = np.arange(0, og_len * 3600)
df_interp = hourly_df.reindex(second_index)

# fills in all the nulls
df_interp = df_interp.interpolate(method='linear', axis=0)

# Normalize to per-second generation rate
expected_counts_seconds = df_interp / 3600

expected_counts_seconds.columns = list(range(1,101))

print(expected_counts_seconds.shape)
expected_counts_seconds

In [ ]:
sns.histplot(expected_counts_seconds[90])

expected_counts_seconds.mean().hist(bins=30)


# Export

In [ ]:
expected_secound_counts.to_csv('data/vehicle_counts/expected_counts_seconds.csv', index=False)

In [ ]:


sns.lineplot(expected_counts_seconds[95])

In [ ]:
from matplotlib.ticker import FuncFormatter

# Plot the lineplot
sns.lineplot(expected_counts_seconds[95])

# Format the x-axis to display hours
ax = plt.gca()
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x // 3600)}am" if x // 3600 < 12 else f"{int(x // 3600 - 12)}pm"))
ax.xaxis.set_major_locator(MultipleLocator(7200))  # Show every other hour (2-hour intervals)

plt.grid()
plt.show()